# FFPP_c23_Baselines_CelebDF_Smoothing_Seed42

새 FF++ c23 Xception·CLIP B/16·DINOv2 B/14의 완료된 best.pt를 평가한다. 학습·UDA 없음.

- **고정 평가:** Celeb-DF V2 공식 테스트 518영상 / 8,288이미지(Real 2,848, Fake 5,440).
- **조건:** 원본, 스무딩 25%·50%·75%. 원본 native 얼굴 crop에 변형한 뒤 모델별 기존 resize·정규화를 적용한다. 추가 JPEG 없음.
- 기존 bilateral + 얼굴 기하 마스크를 그대로 사용한다. Instagram 공식 필터 구현은 아니다. 얼굴 검출실패 이미지는 변형하지 않고 포함한다.
- **지표:** 이미지 단위 AUC·Accuracy·F1·Real FPR·Fake FNR. threshold 0.5 고정. video 성능 지표 없음.
- 평가 FP32, 모델별 입력 조건 유지. 테스트로 모델/임계값/강도를 선택하지 않는다.
- 중단 후 같은 셀을 재실행하면 완료된 모델·조건의 결과를 재사용한다. 완료되지 않은 조건만 다시 평가한다.
- 같은 인물은 허용하지만 평가 영상/프레임은 향후 UDA에서 제외한다. 원본–조작 영상 연결 중복은 UDA 준비 시 별도 확인해야 한다.

## Drive 준비

기존 `MyDrive/instagram-deepfake-robustness/data/celebdf_v2_face_crops16_v1/celebdf_v2_roi16_test_v1.tar`만 필요하다.
FF++ 데이터 TAR와 Celeb-DF non-test TAR는 **이 평가에서 필요 없다**.
모델은 기존 `runs/ffpp_c23_4methods_baseline_v1/` 아래 완료된 run을 사용한다.
먼저 새 코드를 GitHub에 push하고 GPU 코랩에서 실행한다.

## Cell 1 — Drive·코드·설치

PIL/torch는 노트북 kernel에서 import하지 않는다. 실제 평가를 별도 Python으로 실행하여 설치 후 kernel 패키지 충돌을 피한다.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, shutil
PROJECT=Path('/content/instagram_deepfake_robustness')
if PROJECT.exists():
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/uyh1556/instagram-deepfake-robustness.git',str(PROJECT)],check=True)
assert (PROJECT/'scripts/evaluate_ffpp_baselines_celebdf_smoothing.py').is_file(),'새 코드 push 필요'
subprocess.run([sys.executable,'-m','pip','install','-q','setuptools==80.9.0','wheel'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation','-r',str(PROJECT/'requirements-ffpp-c23-smoothing-eval.txt')],check=True)
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(),"GPU 런타임 선택"; print(torch.cuda.get_device_name(0))'],check=True)




## Cell 2 — 변수

아직 학습 중인 모델은 MODELS에서 빼고 완료 후 추가하면 된다. 세 모델을 모두 평가하려면 그대로 실행한다.



In [ ]:
MODELS=['xception','clip_b16','dinov2_b14']
ROOT=Path('/content/drive/MyDrive/instagram-deepfake-robustness')
DRIVE_DATA=ROOT/'data/celebdf_v2_face_crops16_v1'
DRIVE_RUNS=ROOT/'runs/ffpp_c23_4methods_baseline_v1'
DATA_ROOT=Path('/content/celebdf_v2_face_crops16_v1')
TEST_MANIFEST=DATA_ROOT/'manifests/celebdf_v2_roi16_test_v1.csv'
RUNS_ROOT=Path('/content/ffpp_c23_baseline_eval_runs')
CACHE_ROOT=Path('/content/celebdf_official_smoothing_cache_v1')
EVALUATION_ROOT=ROOT/'evaluations/ffpp_c23_baselines_celebdf_official_smoothing_v1/seed42'
MODEL_ARGS=' '.join(MODELS)
COMMON=f'--models {MODEL_ARGS} --seed 42 --data-root "{DATA_ROOT}" --manifest "{TEST_MANIFEST}" --cache-root "{CACHE_ROOT}" --runs-root "{RUNS_ROOT}" --output-root "{EVALUATION_ROOT}" --batch-size 32 --workers 2'
print('Models:',MODELS,'Output:',EVALUATION_ROOT)




## Cell 3 — 테스트 TAR·완료된 모델 준비

TAR와 체크포인트를 로컬로 복사하여 Drive에서 매 batch 읽지 않는다. 이미지 전체 해시 검사는 하지 않는다.



In [ ]:
archive=DRIVE_DATA/'celebdf_v2_roi16_test_v1.tar'
assert archive.is_file(),archive
marker=Path('/content/.celebdf_official_roi16_test_v1.done')
if not marker.exists() or not TEST_MANIFEST.is_file():
    local=Path('/content/celebdf_v2_roi16_test_v1.tar')
    print('Copying official test TAR...',flush=True)
    shutil.copy2(archive,local)
    assert local.stat().st_size==557015040,'Test TAR 크기 불일치: 업로드 완료 확인'
    print('Extracting official test...',flush=True)
    result=subprocess.run(['tar','-xf',str(local),'-C','/content'],capture_output=True,text=True)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError('압축 해제 실패: 위 오류 확인')
    marker.touch()
    local.unlink()
for model in MODELS:
    name=f'{model}_ffpp_c23_roi16_baseline_v1_seed42'
    source=DRIVE_RUNS/name
    dest=RUNS_ROOT/name
    for filename in ['best.pt','config.json','completed.json']:
        assert (source/filename).is_file(),f'학습 완료 확인: {source/filename}'
    dest.mkdir(parents=True,exist_ok=True)
    for filename in ['best.pt','config.json','completed.json']:
        shutil.copy2(source/filename,dest/filename)
    print('Ready:',model,flush=True)




## Cell 4 — 스무딩 준비

원본을 덮어쓰지 않는다. 세 강도의 PNG cache를 한 번 생성하여 모든 모델이 공유한다.



In [ ]:
!python -u "{PROJECT}/scripts/evaluate_ffpp_baselines_celebdf_smoothing.py" prepare {COMMON}
assert _exit_code==0,'스무딩 준비 실패: 위 오류 확인'




## Cell 5 — 원본·스무딩 평가

모델별 4조건, 총 12번 추론. batch 진행률·경과시간·ETA와 조건당 요약 한 줄만 표시한다.
완료된 baseline 결과는 이후 동일 평가 조건의 augmentation·UDA 비교에 그대로 재사용한다.



In [ ]:
!python -u "{PROJECT}/scripts/evaluate_ffpp_baselines_celebdf_smoothing.py" evaluate {COMMON}
assert _exit_code==0,'평가 실패: 위 오류 확인 후 Cell 5 재실행'




## Cell 6 — 표로 결과 확인

FPR·FNR·Accuracy는 아래 표에서 %로 표시한다. CSV 원본은 0~1 비율이다.



In [ ]:
import csv
from IPython.display import Markdown, display
rows=list(csv.DictReader((EVALUATION_ROOT/'condition_summary.csv').open()))
table=['|Model|Condition|AUC|Accuracy|F1|Real FPR|Fake FNR|',
       '|---|---|---:|---:|---:|---:|---:|']
for r in rows:
    table.append(f"|{r['model']}|{r['condition']}|{float(r['auc']):.4f}|{float(r['accuracy']):.2%}|{float(r['f1']):.4f}|{float(r['real_fpr']):.2%}|{float(r['fake_fnr']):.2%}|")
display(Markdown('\n'.join(table)))
print('Saved:',EVALUATION_ROOT)
print('condition_summary.csv: 전체 지표 / paired_deltas.csv: 원본 대비 변화')
print('evaluation_manifest.csv, evaluation_protocol.json: 이후 공통 평가·UDA 제외 목록')
